# HuskySwap student: Qwen3-4B on your own GPU

Serves the student model with vLLM behind a Cloudflare tunnel, so `npm run eval -- --model=student` (or the app) can use it.

* **Untrained baseline:** leave `ADAPTER` empty. This is the "before training" score.
* **Fine-tuned student (later):** set `ADAPTER` to the LoRA folder (e.g. on Google Drive) and `VERSION` to its tag.

**Runtime:** **L4** is enough (the 4B model needs ~8 GB); A100 or G4 also work. This can run in a second Colab session next to the teacher notebook.

**Security:** only requests with the generated API key are answered. Put it in `.env.local` yourself. Stop the runtime when done.

In [ ]:
# 1. Settings
MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # the student (non-thinking Qwen3-4B)
ADAPTER = ""        # LoRA adapter folder for a fine-tuned student; empty = untrained base model
VERSION = "base"     # recorded on every run (STUDENT_VERSION), e.g. "sft-stage2-r1"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install vLLM (a few minutes)
!pip install -q --upgrade vllm
!python -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# 3. Start the server (downloads ~8 GB the first time)
import secrets, subprocess, time, urllib.request

API_KEY = secrets.token_urlsafe(24)
served_name = MODEL if not ADAPTER else f"huskyswap-student-{VERSION}"
cmd = ["vllm", "serve", MODEL,
       "--host", "127.0.0.1", "--port", "8001",
       "--api-key", API_KEY,
       "--max-model-len", "32768",
       "--gpu-memory-utilization", "0.90",
       "--enable-auto-tool-choice", "--tool-call-parser", "hermes"]
if ADAPTER:
    cmd += ["--enable-lora", "--max-lora-rank", "64", "--lora-modules", f"{served_name}={ADAPTER}"]
log = open("/content/vllm-student.log", "w")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

start = time.time()
while True:
    if server.poll() is not None:
        raise RuntimeError("vLLM exited; read /content/vllm-student.log:\n" + open("/content/vllm-student.log").read()[-3000:])
    try:
        urllib.request.urlopen("http://127.0.0.1:8001/health", timeout=5)
        break
    except Exception:
        time.sleep(10)
        print(f"loading... {int(time.time() - start)} s", end="\r")
print(f"\nserver ready after {int(time.time() - start)} s, serving {served_name}")

In [ ]:
# 4. Tunnel + the settings for .env.local
import re, socket
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tunnel_log = open("/content/tunnel-student.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8001"], stdout=tunnel_log, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(2)
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel-student.log").read())
    if found:
        url = found.group(0)
        break
if url:
    # A new quick-tunnel address takes a few seconds to a minute to appear in DNS
    host = url.removeprefix("https://")
    for _ in range(45):
        try:
            socket.gethostbyname(host)
            break
        except socket.gaierror:
            time.sleep(2)
    else:
        print("warning: the tunnel address does not resolve yet; wait a minute before using it")
if not url:
    raise RuntimeError("No tunnel address yet; read /content/tunnel-student.log")
print("Add these to .env.local:\n")
print(f"STUDENT_BASE_URL={url}/v1")
print(f"STUDENT_API_KEY={API_KEY}")
print(f"STUDENT_MODEL={served_name}")
print(f"STUDENT_VERSION={VERSION}")

In [ ]:
# 5. Test a tool call through the tunnel
import json, urllib.error
body = {
    "model": served_name,
    "messages": [{"role": "user", "content": "How many listings are there? Use the tool."}],
    "tools": [{"type": "function", "function": {"name": "run_sql", "description": "Run a SQL query",
               "parameters": {"type": "object", "properties": {"sql": {"type": "string"}}, "required": ["sql"]}}}],
    "tool_choice": "auto", "max_tokens": 512,
}
req = urllib.request.Request(f"{url}/v1/chat/completions", data=json.dumps(body).encode(),
                             headers={"content-type": "application/json", "authorization": f"Bearer {API_KEY}"})
t0 = time.time()
for attempt in range(12):  # the tunnel can take a minute to become reachable
    try:
        reply = json.load(urllib.request.urlopen(req, timeout=120))
        break
    except urllib.error.URLError as e:
        print(f"not reachable yet ({e.reason}); retrying...")
        time.sleep(5)
else:
    raise RuntimeError("tunnel still unreachable; read the tunnel log")
print(f"{time.time() - t0:.2f} s | tool calls: {json.dumps(reply['choices'][0]['message'].get('tool_calls'))[:300]}")

In [ ]:
# 6. If something looks wrong: the last lines of the server log
print(open("/content/vllm-student.log").read()[-4000:])